## Section 0 — Imports, Configuration, QuantBook Init & Cache Helpers

This notebook extracts **IV term-structure slope** and **Volatility Risk Premium (VRP)** features from the already-cached SPY option chain in the GEX Object Store. These features are then fed into an HMM regime model to test whether they sharpen regime separation, and evaluated as standalone VRP trading signals.

All data reuse leverages the `gex_research/` namespace — no new `History[OptionUniverse]` call is needed.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 0 — Imports, Configuration, QuantBook Init & Cache Helpers
# ═══════════════════════════════════════════════════════════════════════════════

# ── Imports ──────────────────────────────────────────────────────────────────
from QuantConnect import *
from QuantConnect.Research import *
from QuantConnect.Data.Custom.CBOE import CBOE
from QuantConnect.Data.Market import TradeBar

import numpy as np
import pandas as pd
from scipy import stats as sp_stats
from sklearn.mixture import GaussianMixture
from hmmlearn.hmm import GaussianHMM

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import warnings
import pickle, io, sys, tracemalloc
from datetime import datetime

# ── Configuration ────────────────────────────────────────────────────────────
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
plt.style.use('dark_background')
plt.rcParams.update({
    "figure.figsize": (14, 5),
    "figure.dpi":     110,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "font.size":      10,
})
np.random.seed(42)

# ── Parameters ───────────────────────────────────────────────────────────────
START_DATE, END_DATE       = datetime(2012, 1, 1), datetime(2024, 12, 31)
UNDERLYING                 = "SPY"
FRONT_DTE_MIN, FRONT_DTE_MAX = 20, 45
BACK_DTE_MIN,  BACK_DTE_MAX  = 46, 90
ATM_MONEYNESS_TOL          = 0.02
RVOL_WINDOW, RVOL_WINDOW_LONG = 21, 63
LOOKBACK_ZSCORE, Z_THRESHOLD  = 63, 1.0
HMM_N_STATES               = 3
OBJ_PREFIX                 = "vrp_research"

# GEX namespace cache keys (from research_Dealer_Gamma_Exposure.ipynb)
GEX_KEY_CHAIN   = "section1_spy_option_chain_v2"
GEX_KEY_GEX     = "section2_gex_daily_v3"
GEX_KEY_SIGNAL  = "section4_signal_v3"

# ── Memory tracking ─────────────────────────────────────────────────────────
tracemalloc.start()

# ── QuantBook Init ───────────────────────────────────────────────────────────
qb = QuantBook()

# ── Cache Helpers (vrp_research namespace) ───────────────────────────────────
def obj_key(name):        return f"{OBJ_PREFIX}/{name}"
def save_to_store(key, obj):
    buf = pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)
    qb.object_store.save_bytes(obj_key(key), buf)
    print(f"  [OK] [ObjectStore] Saved '{key}' ({len(buf)/1e6:.2f} MB)")
def load_from_store(key):
    obj = pickle.loads(bytes(qb.object_store.read_bytes(obj_key(key))))
    print(f"  [OK] [ObjectStore] Loaded '{key}'"); return obj
def store_exists(key):    return qb.object_store.contains_key(obj_key(key))

# Cross-namespace reader for GEX notebook caches
def load_from_gex(key):
    obj = pickle.loads(bytes(qb.object_store.read_bytes(f"gex_research/{key}")))
    print(f"  [OK] [GEX store] Loaded '{key}'"); return obj
def gex_exists(key):      return qb.object_store.contains_key(f"gex_research/{key}")

# ── Environment Summary ─────────────────────────────────────────────────────
print("═" * 70)
print("  VOLATILITY TERM STRUCTURE & VRP RESEARCH NOTEBOOK")
print("═" * 70)
print(f"  NumPy {np.__version__}  |  Pandas {pd.__version__}")
print(f"  Date range : {START_DATE.date()} → {END_DATE.date()}")
print(f"  Underlying : {UNDERLYING}")
print(f"  Front DTE  : {FRONT_DTE_MIN}–{FRONT_DTE_MAX}")
print(f"  Back DTE   : {BACK_DTE_MIN}–{BACK_DTE_MAX}")
print(f"  ATM tol    : ±{ATM_MONEYNESS_TOL:.0%}")
print(f"  HMM states : {HMM_N_STATES}")
print(f"  OBJ_PREFIX : {OBJ_PREFIX}")
print("═" * 70)

peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"  [OK] Section 0 complete | peak RAM: {peak_mem:.1f} MB")

## Section 1 — Load Existing Caches & Supplementary Data

The SPY option chain (~2M+ rows with IV, gamma, delta, OI, DTE per strike) is already cached from the GEX notebook. Loading it directly avoids re-running the hours-long `History[OptionUniverse]` call.

We supplement with daily SPY prices (for realized vol computation) and the CBOE VIX index (for the variance risk premium benchmark) over the full 2012–2024 window.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — Load Existing Caches + Supplementary Data
# ═══════════════════════════════════════════════════════════════════════════════

# ── 1a. Load GEX caches ─────────────────────────────────────────────────────
chain_df  = load_from_gex(GEX_KEY_CHAIN)
daily_gex = load_from_gex(GEX_KEY_GEX)
signal_df = load_from_gex(GEX_KEY_SIGNAL)

# ── 1b. SPY daily prices ────────────────────────────────────────────────────
CACHE_KEY_S1_SPY = "section1_spy_prices_v1"

if store_exists(CACHE_KEY_S1_SPY):
    spy_prices = load_from_store(CACHE_KEY_S1_SPY)
else:
    spy_equity = qb.add_equity(UNDERLYING, Resolution.DAILY)
    spy_equity.set_data_normalization_mode(DataNormalizationMode.RAW)
    spy_hist = qb.history(spy_equity.symbol, START_DATE, END_DATE, Resolution.DAILY)
    spy_prices = spy_hist['close'].droplevel(0).to_frame('close')
    spy_prices.index = pd.DatetimeIndex(spy_prices.index.date)
    spy_prices.index.name = 'date'
    save_to_store(CACHE_KEY_S1_SPY, spy_prices)

# ── 1c. VIX history ─────────────────────────────────────────────────────────
CACHE_KEY_S1_VIX = "section1_vix_history_v1"

if store_exists(CACHE_KEY_S1_VIX):
    vix_series = load_from_store(CACHE_KEY_S1_VIX)
else:
    vix_symbol = qb.add_data(CBOE, "VIX", Resolution.DAILY).symbol
    vix_hist = qb.history(vix_symbol, START_DATE, END_DATE, Resolution.DAILY)
    vix_series = vix_hist['close'].droplevel(0).rename('vix')
    vix_series.index = pd.DatetimeIndex(vix_series.index.date)
    vix_series.index.name = 'date'
    save_to_store(CACHE_KEY_S1_VIX, vix_series)

# ── 1d. Data summary ────────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  DATA SUMMARY")
print("═" * 70)
datasets = {
    'chain_df':  chain_df,
    'daily_gex': daily_gex,
    'signal_df': signal_df,
    'spy_prices': spy_prices,
    'vix_series': vix_series if isinstance(vix_series, pd.DataFrame) else vix_series.to_frame(),
}
print(f"  {'Dataset':<14} {'Rows':>10} {'Cols':>6} {'Start':>12} {'End':>12} {'MB':>8}")
print(f"  {'─'*14} {'─'*10} {'─'*6} {'─'*12} {'─'*12} {'─'*8}")
for name, df_ in datasets.items():
    n = len(df_)
    c = df_.shape[1] if hasattr(df_, 'shape') and len(df_.shape) > 1 else 1
    s = str(df_.index.min())[:10]
    e = str(df_.index.max())[:10]
    mb = df_.memory_usage(deep=True).sum() / 1e6 if hasattr(df_, 'memory_usage') else 0
    print(f"  {name:<14} {n:>10,} {c:>6} {s:>12} {e:>12} {mb:>8.1f}")
print("═" * 70)

peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"  [OK] Section 1 complete | peak RAM: {peak_mem:.1f} MB")

In [ ]:
# ── Section 1 Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Panel 1: SPY price
axes[0].plot(spy_prices.index, spy_prices['close'], color='white', linewidth=0.8)
axes[0].set_title('SPY Daily Close')
axes[0].set_ylabel('Price ($)')
axes[0].grid(alpha=0.3)

# Panel 2: VIX with horizontal line at 20
vix_idx = vix_series.index if isinstance(vix_series, pd.Series) else vix_series.index
vix_vals = vix_series.values if isinstance(vix_series, pd.Series) else vix_series['vix'].values
axes[1].plot(vix_idx, vix_vals, color='cyan', linewidth=0.8)
axes[1].axhline(20, color='red', linestyle='--', alpha=0.5, label='VIX = 20')
axes[1].set_title('CBOE VIX Index')
axes[1].set_ylabel('VIX Level')
axes[1].legend(loc='upper right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Data coverage: chain_df starts ~2020 (options cache); SPY/VIX extend back to 2012.")
print("Overlap period used for VRP computation: chain_df date range.")

## Section 2 — IV Term-Structure Extraction (Vectorized)

The IV term-structure slope is defined as **front-month ATM IV minus back-month ATM IV**. An inverted curve (slope > 0) indicates near-term stress priced in and expected to pass. A steep curve (slope < 0) signals long-dated uncertainty.

Campasano & Linn (2017) show that term-structure state predicts the VRP available at each maturity — the regime dictates whether short-vol strategies earn or lose money.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — IV Term-Structure Extraction (Vectorized)
# ═══════════════════════════════════════════════════════════════════════════════

CACHE_KEY_S2 = "section2_iv_term_structure_v1"

if store_exists(CACHE_KEY_S2):
    iv_term_df = load_from_store(CACHE_KEY_S2)
else:
    # ── ATM + tenor masks (fully vectorized) ─────────────────────────────────
    atm_mask   = chain_df['moneyness'].between(1 - ATM_MONEYNESS_TOL, 1 + ATM_MONEYNESS_TOL)
    front_mask = chain_df['dte'].between(FRONT_DTE_MIN, FRONT_DTE_MAX) & atm_mask
    back_mask  = chain_df['dte'].between(BACK_DTE_MIN,  BACK_DTE_MAX)  & atm_mask

    # ── Median IV per date for each tenor bucket ─────────────────────────────
    iv_front = chain_df.loc[front_mask].groupby('date')['iv'].median().rename('iv_front')
    iv_back  = chain_df.loc[back_mask ].groupby('date')['iv'].median().rename('iv_back')

    iv_term_df = pd.concat([iv_front, iv_back], axis=1).dropna()

    # ── Derived features ─────────────────────────────────────────────────────
    iv_term_df['ts_slope']    = iv_term_df['iv_front'] - iv_term_df['iv_back']
    iv_term_df['ts_ratio']    = iv_term_df['iv_front'] / iv_term_df['iv_back']

    # Rolling 63-day z-score of ts_slope
    roll_mean = iv_term_df['ts_slope'].rolling(LOOKBACK_ZSCORE, min_periods=LOOKBACK_ZSCORE).mean()
    roll_std  = iv_term_df['ts_slope'].rolling(LOOKBACK_ZSCORE, min_periods=LOOKBACK_ZSCORE).std()
    iv_term_df['ts_slope_z']  = (iv_term_df['ts_slope'] - roll_mean) / roll_std

    iv_term_df['ts_inverted'] = (iv_term_df['ts_slope'] > 0).astype(np.int8)

    # 3-state regime (vectorized np.select)
    # 2 = Inverted/Crisis, 0 = Steep/Calm, 1 = Normal
    iv_term_df['ts_regime'] = np.select(
        [iv_term_df['ts_slope_z'] > 1.0, iv_term_df['ts_slope_z'] < -1.0],
        [2, 0], default=1).astype(np.int8)

    iv_term_df.index = pd.DatetimeIndex(iv_term_df.index)
    iv_term_df.index.name = 'date'

    save_to_store(CACHE_KEY_S2, iv_term_df)

# ── Assertions ───────────────────────────────────────────────────────────────
assert iv_term_df['iv_front'].between(0.01, 2.0).all(), "iv_front out of range"
assert iv_term_df['iv_back'].between(0.01, 2.0).all(),  "iv_back out of range"
assert len(iv_term_df) > 200, f"Only {len(iv_term_df)} rows — expected > 200"

# ── Summary statistics ───────────────────────────────────────────────────────
regime_counts = iv_term_df['ts_regime'].value_counts(normalize=True).sort_index()
regime_names = {0: 'Steep/Calm', 1: 'Normal', 2: 'Inverted/Crisis'}

print("\n" + "═" * 70)
print("  IV TERM STRUCTURE SUMMARY")
print("═" * 70)
print(f"  Total trading days: {len(iv_term_df):,}")
for r, pct in regime_counts.items():
    mean_iv_f = iv_term_df.loc[iv_term_df['ts_regime'] == r, 'iv_front'].mean()
    mean_iv_b = iv_term_df.loc[iv_term_df['ts_regime'] == r, 'iv_back'].mean()
    print(f"  {regime_names.get(r, r):<20} {pct:>6.1%}  |  mean IV front={mean_iv_f:.3f}  back={mean_iv_b:.3f}")
print("═" * 70)

peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"  [OK] Section 2 complete | peak RAM: {peak_mem:.1f} MB")

In [ ]:
# ── Section 2 Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Panel 1: IV front + back over time with regime shading
ax = axes[0]
ax.plot(iv_term_df.index, iv_term_df['iv_front'], color='cyan', linewidth=0.8, label='IV Front (20-45 DTE)')
ax.plot(iv_term_df.index, iv_term_df['iv_back'],  color='white', linewidth=0.8, label='IV Back (46-90 DTE)')
# Shade inverted (red) and steep (green)
inverted_mask = iv_term_df['ts_regime'] == 2
steep_mask    = iv_term_df['ts_regime'] == 0
for mask, color, lbl in [(inverted_mask, 'red', 'Inverted'), (steep_mask, 'green', 'Steep')]:
    idx = iv_term_df.index[mask]
    for i in range(len(idx)):
        ax.axvspan(idx[i], idx[i] + pd.Timedelta(days=1), alpha=0.15, color=color, linewidth=0)
ax.set_title('ATM Implied Volatility: Front vs Back Month')
ax.set_ylabel('IV')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

# Panel 2: ts_slope bar chart
ax = axes[1]
colors = np.where(iv_term_df['ts_slope'] > 0, 'red', 'green')
ax.bar(iv_term_df.index, iv_term_df['ts_slope'], color=colors, width=1.5, alpha=0.7)
ax.axhline(0, color='white', linewidth=0.5)
ax.set_title('Term Structure Slope (Front − Back)')
ax.set_ylabel('Slope')
# Annotate key events
for dt_str, label in [('2020-03-16', 'COVID'), ('2024-08-05', 'Aug-2024')]:
    dt = pd.Timestamp(dt_str)
    if dt in iv_term_df.index:
        ax.annotate(label, xy=(dt, iv_term_df.loc[dt, 'ts_slope']),
                    fontsize=9, color='yellow', ha='center',
                    arrowprops=dict(arrowstyle='->', color='yellow'))
ax.grid(alpha=0.3)

# Panel 3: ts_slope histogram colored by regime
ax = axes[2]
regime_colors = {0: 'green', 1: 'gray', 2: 'red'}
for r in [0, 1, 2]:
    mask = iv_term_df['ts_regime'] == r
    vals = iv_term_df.loc[mask, 'ts_slope'].dropna()
    ax.hist(vals, bins=50, alpha=0.5, color=regime_colors[r],
            label=f"{'Steep' if r==0 else 'Normal' if r==1 else 'Inverted'}")
mean_slope = iv_term_df['ts_slope'].mean()
std_slope  = iv_term_df['ts_slope'].std()
ax.axvline(mean_slope - std_slope, color='white', linestyle='--', alpha=0.5)
ax.axvline(mean_slope + std_slope, color='white', linestyle='--', alpha=0.5)
ax.set_title('Term Structure Slope Distribution by Regime')
ax.set_xlabel('Slope')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretation: Inverted term structure (red) clusters around stress events.")
print("Steep curve (green) indicates calm periods with elevated long-dated uncertainty.")

## Section 3 — Volatility Risk Premium (VRP)

VRP = Implied Volatility − Realized Volatility. A positive VRP means options are overpriced relative to actual moves — the premium available to vol sellers. The VRP inverts in crises when realized vol exceeds implied.

Front-month VRP uses 21-day realized vol; back-month uses 63-day realized vol. Guo & Loeper (2020) demonstrate that delta-hedged selling is robust when VRP > 0 and the term structure is not inverted.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Volatility Risk Premium (VRP)
# ═══════════════════════════════════════════════════════════════════════════════

CACHE_KEY_S3 = "section3_vrp_features_v1"

if store_exists(CACHE_KEY_S3):
    vrp_df = load_from_store(CACHE_KEY_S3)
else:
    # ── Realized volatility (vectorized) ─────────────────────────────────────
    spy_log_ret = np.log(spy_prices['close']).diff()
    rvol_21 = (spy_log_ret.rolling(RVOL_WINDOW).std() * np.sqrt(252)).rename('rvol_21')
    rvol_63 = (spy_log_ret.rolling(RVOL_WINDOW_LONG).std() * np.sqrt(252)).rename('rvol_63')

    # ── Join IV term structure with realized vol ─────────────────────────────
    vrp_df = iv_term_df.join(rvol_21).join(rvol_63)

    # ── VRP features ─────────────────────────────────────────────────────────
    vrp_df['vrp_front']    = vrp_df['iv_front'] - vrp_df['rvol_21']
    vrp_df['vrp_back']     = vrp_df['iv_back']  - vrp_df['rvol_63']
    vrp_df['vrp_spread']   = vrp_df['vrp_front'] - vrp_df['vrp_back']

    # Rolling 63-day z-scores
    for col in ['vrp_front', 'vrp_back', 'vrp_spread']:
        rm = vrp_df[col].rolling(LOOKBACK_ZSCORE, min_periods=LOOKBACK_ZSCORE).mean()
        rs = vrp_df[col].rolling(LOOKBACK_ZSCORE, min_periods=LOOKBACK_ZSCORE).std()
        vrp_df[f'{col}_z'] = (vrp_df[col] - rm) / rs

    # ── Join GEX features ────────────────────────────────────────────────────
    vrp_df = vrp_df.join(signal_df[['net_gex', 'gex_zscore']], how='left')
    vrp_df['gex_regime'] = np.sign(vrp_df['net_gex']).fillna(0).astype(np.int8)

    vrp_df.index = pd.DatetimeIndex(vrp_df.index)
    vrp_df.index.name = 'date'

    save_to_store(CACHE_KEY_S3, vrp_df)

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  VRP SUMMARY")
print("═" * 70)

# Mean VRP by ts_regime
regime_names = {0: 'Steep/Calm', 1: 'Normal', 2: 'Inverted/Crisis'}
print(f"  {'Regime':<20} {'VRP Front':>10} {'VRP Back':>10} {'Count':>8}")
print(f"  {'─'*20} {'─'*10} {'─'*10} {'─'*8}")
for r in sorted(vrp_df['ts_regime'].dropna().unique()):
    mask = vrp_df['ts_regime'] == r
    mf = vrp_df.loc[mask, 'vrp_front'].mean()
    mb = vrp_df.loc[mask, 'vrp_back'].mean()
    cnt = mask.sum()
    print(f"  {regime_names.get(int(r), str(r)):<20} {mf:>10.4f} {mb:>10.4f} {cnt:>8,}")

pct_pos = (vrp_df['vrp_front'] > 0).mean()
print(f"\n  % days VRP_front > 0: {pct_pos:.1%}")

# Correlation matrix
corr_cols = ['vrp_front', 'vrp_back', 'ts_slope', 'gex_zscore']
valid = vrp_df[corr_cols].dropna()
if len(valid) > 50:
    corr = valid.corr()
    print(f"\n  Correlation matrix ({len(valid)} obs):")
    print(corr.to_string(float_format=lambda x: f"{x:>7.3f}"))
print("═" * 70)

peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"  [OK] Section 3 complete | peak RAM: {peak_mem:.1f} MB")

In [ ]:
# ── Section 3 Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Panel 1: VRP front + back over time
ax = axes[0]
ax.plot(vrp_df.index, vrp_df['vrp_front'], color='cyan', linewidth=0.8, label='VRP Front (IV−RV21)')
ax.plot(vrp_df.index, vrp_df['vrp_back'],  color='orange', linewidth=0.8, label='VRP Back (IV−RV63)')
ax.axhline(0, color='white', linewidth=0.5)
# Shade VRP > 0 green, < 0 red
ax.fill_between(vrp_df.index, 0, vrp_df['vrp_front'],
                where=vrp_df['vrp_front'] > 0, color='green', alpha=0.15)
ax.fill_between(vrp_df.index, 0, vrp_df['vrp_front'],
                where=vrp_df['vrp_front'] < 0, color='red', alpha=0.15)
ax.set_title('Volatility Risk Premium: Front & Back Month')
ax.set_ylabel('VRP')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

# Panel 2: VRP spread over time
ax = axes[1]
ax.plot(vrp_df.index, vrp_df['vrp_spread'], color='magenta', linewidth=0.8)
ax.axhline(0, color='white', linewidth=0.5)
ax.set_title('VRP Spread (Front − Back)')
ax.set_ylabel('Spread')
ax.grid(alpha=0.3)

# Panel 3: 2×2 scatter matrix
ax = axes[2]
scatter_cols = ['vrp_front', 'vrp_back', 'ts_slope', 'gex_zscore']
valid = vrp_df[scatter_cols + ['ts_regime']].dropna()
regime_colors = {0: 'green', 1: 'gray', 2: 'red'}
if len(valid) > 50:
    for r in sorted(valid['ts_regime'].unique()):
        mask = valid['ts_regime'] == r
        ax.scatter(valid.loc[mask, 'vrp_front'], valid.loc[mask, 'ts_slope'],
                   c=regime_colors.get(int(r), 'white'), alpha=0.2, s=10,
                   label=f"Regime {int(r)}")
    r_val = valid['vrp_front'].corr(valid['ts_slope'])
    ax.annotate(f'r = {r_val:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
                fontsize=11, color='yellow')
    ax.set_xlabel('VRP Front')
    ax.set_ylabel('TS Slope')
    ax.set_title('VRP Front vs Term Structure Slope (colored by regime)')
    ax.legend(loc='upper right', markerscale=3)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretation: VRP is mostly positive (options overpriced), but inverts during crises.")
print("The VRP spread captures relative richness across the term structure.")

## Section 4 — HMM Regime Augmentation

The research2.ipynb HMM used `[vix_z, vix_ret, rvol_21]` as features. Here we add `ts_slope_z` (shape of vol expectations), `vrp_front_z`, and `gex_zscore` (orthogonal dealer positioning signal).

**Goal:** Sharpen Crisis vs Elevated distinction — especially distinguishing "high VIX + dealers long gamma + steep curve" (dip-buying opportunity) from "high VIX + inverted + dealers short gamma" (momentum amplification / true crisis).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — HMM Regime Augmentation
# ═══════════════════════════════════════════════════════════════════════════════

CACHE_KEY_S4 = "section4_hmm_augmented_v1"

if store_exists(CACHE_KEY_S4):
    hmm_results = load_from_store(CACHE_KEY_S4)
    hmm_features   = hmm_results['hmm_features']
    hmm_a_states   = hmm_results['hmm_a_states']
    hmm_b_states   = hmm_results['hmm_b_states']
    model_a_info   = hmm_results['model_a_info']
    model_b_info   = hmm_results['model_b_info']
else:
    # ── Build feature matrix ─────────────────────────────────────────────────
    vix_s = vix_series if isinstance(vix_series, pd.Series) else vix_series['vix']
    vix_s.index = pd.DatetimeIndex(vix_s.index)

    # VIX z-score (rolling 252d)
    vix_rm = vix_s.rolling(252, min_periods=126).mean()
    vix_rs = vix_s.rolling(252, min_periods=126).std()
    vix_z  = ((vix_s - vix_rm) / vix_rs).rename('vix_z')

    # VIX daily return
    vix_ret = vix_s.pct_change().rename('vix_ret')

    # Realized vol
    spy_log_ret = np.log(spy_prices['close']).diff()
    rvol_21_full = (spy_log_ret.rolling(RVOL_WINDOW).std() * np.sqrt(252)).rename('rvol_21')

    # Combine all features
    hmm_features = pd.concat([
        vix_z, vix_ret, rvol_21_full,
        vrp_df['ts_slope_z'], vrp_df['vrp_front_z'], vrp_df['gex_zscore']
    ], axis=1)
    hmm_features.columns = ['vix_z', 'vix_ret', 'rvol_21', 'ts_slope_z', 'vrp_front_z', 'gex_z']
    hmm_features['gex_z'] = hmm_features['gex_z'].fillna(0)  # neutral before 2020

    # Drop NaN on baseline features — keep full history
    hmm_features = hmm_features.dropna(subset=['vix_z', 'vix_ret', 'rvol_21', 'ts_slope_z'])
    hmm_features.index = pd.DatetimeIndex(hmm_features.index)

    # ── Model A: baseline [vix_z, vix_ret, rvol_21] ─────────────────────────
    feat_a = hmm_features[['vix_z', 'vix_ret', 'rvol_21']].values.astype(np.float64)
    T_train = int(len(feat_a) * 0.8)

    model_a = GaussianHMM(n_components=HMM_N_STATES, covariance_type='full',
                           n_iter=100, random_state=42)
    model_a.fit(feat_a[:T_train])
    states_a_raw = model_a.predict(feat_a)

    # Sort states by mean vix_z: 0=Calm, 1=Elevated, 2=Crisis
    state_means_a = np.array([feat_a[states_a_raw == s, 0].mean() for s in range(HMM_N_STATES)])
    sort_order_a = np.argsort(state_means_a)
    remap_a = np.zeros(HMM_N_STATES, dtype=int)
    for new_label, old_label in enumerate(sort_order_a):
        remap_a[old_label] = new_label
    hmm_a_states = remap_a[states_a_raw]

    # ── Model B: augmented [vix_z, vix_ret, rvol_21, ts_slope_z, vrp_front_z, gex_z]
    feat_b = hmm_features[['vix_z', 'vix_ret', 'rvol_21', 'ts_slope_z', 'vrp_front_z', 'gex_z']].values.astype(np.float64)

    model_b = GaussianHMM(n_components=HMM_N_STATES, covariance_type='full',
                           n_iter=100, random_state=42)
    model_b.fit(feat_b[:T_train])
    states_b_raw = model_b.predict(feat_b)

    # Sort states by mean vix_z
    state_means_b = np.array([feat_b[states_b_raw == s, 0].mean() for s in range(HMM_N_STATES)])
    sort_order_b = np.argsort(state_means_b)
    remap_b = np.zeros(HMM_N_STATES, dtype=int)
    for new_label, old_label in enumerate(sort_order_b):
        remap_b[old_label] = new_label
    hmm_b_states = remap_b[states_b_raw]

    # ── Per-state statistics ─────────────────────────────────────────────────
    spy_ret_aligned = spy_log_ret.reindex(hmm_features.index)
    fwd_5d = spy_ret_aligned.rolling(5).sum().shift(-5)  # next 5-day return

    def model_info(states, feat, label):
        info = {}
        for s in range(HMM_N_STATES):
            mask = states == s
            info[s] = {
                'freq': mask.mean(),
                'mean_vix_z': feat[mask, 0].mean() if mask.any() else np.nan,
                'mean_ts_slope_z': hmm_features.loc[hmm_features.index[mask], 'ts_slope_z'].mean() if mask.any() else np.nan,
                'mean_gex_z': hmm_features.loc[hmm_features.index[mask], 'gex_z'].mean() if mask.any() else np.nan,
                'mean_5d_ret': fwd_5d.iloc[np.where(mask)[0]].mean() if mask.any() else np.nan,
            }
        return info

    model_a_info = model_info(hmm_a_states, feat_a, 'A')
    model_b_info = model_info(hmm_b_states, feat_b, 'B')

    # ── Cache ────────────────────────────────────────────────────────────────
    hmm_results = {
        'hmm_features': hmm_features,
        'hmm_a_states': hmm_a_states,
        'hmm_b_states': hmm_b_states,
        'model_a_info': model_a_info,
        'model_b_info': model_b_info,
    }
    save_to_store(CACHE_KEY_S4, hmm_results)

# ── Print comparison table ───────────────────────────────────────────────────
state_names = {0: 'Calm', 1: 'Elevated', 2: 'Crisis'}
print("\n" + "═" * 70)
print("  HMM REGIME COMPARISON: MODEL A (baseline) vs MODEL B (augmented)")
print("═" * 70)
print(f"  {'State':<10} {'Model':>6} {'Freq%':>7} {'vix_z':>8} {'ts_slope_z':>11} {'gex_z':>7} {'5d ret':>8}")
print(f"  {'─'*10} {'─'*6} {'─'*7} {'─'*8} {'─'*11} {'─'*7} {'─'*8}")
for s in range(HMM_N_STATES):
    for mdl, info in [('A', model_a_info), ('B', model_b_info)]:
        i = info[s]
        print(f"  {state_names[s]:<10} {mdl:>6} {i['freq']:>6.1%} {i['mean_vix_z']:>8.3f} "
              f"{i['mean_ts_slope_z']:>11.3f} {i['mean_gex_z']:>7.3f} {i['mean_5d_ret']:>8.4f}")
print("═" * 70)

peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"  [OK] Section 4 complete | peak RAM: {peak_mem:.1f} MB")

In [ ]:
# ── Section 4 Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(4, 1, figsize=(14, 16))

spy_aligned = spy_prices['close'].reindex(hmm_features.index)
regime_colors = {0: '#2ca02c', 1: '#ff7f0e', 2: '#d62728'}  # green/orange/red
state_names = {0: 'Calm', 1: 'Elevated', 2: 'Crisis'}

# Panel 1: SPY with Model A shading
ax = axes[0]
ax.plot(hmm_features.index, spy_aligned, color='white', linewidth=0.7)
for s in range(HMM_N_STATES):
    mask = hmm_a_states == s
    idx = hmm_features.index[mask]
    for i in range(len(idx)):
        ax.axvspan(idx[i], idx[i] + pd.Timedelta(days=1),
                   alpha=0.2, color=regime_colors[s], linewidth=0)
ax.set_title('SPY with Model A Regime Shading (baseline: vix_z, vix_ret, rvol_21)')
ax.set_ylabel('SPY Price')
ax.grid(alpha=0.3)

# Panel 2: SPY with Model B shading
ax = axes[1]
ax.plot(hmm_features.index, spy_aligned, color='white', linewidth=0.7)
for s in range(HMM_N_STATES):
    mask = hmm_b_states == s
    idx = hmm_features.index[mask]
    for i in range(len(idx)):
        ax.axvspan(idx[i], idx[i] + pd.Timedelta(days=1),
                   alpha=0.2, color=regime_colors[s], linewidth=0)
ax.set_title('SPY with Model B Regime Shading (augmented: +ts_slope_z, vrp_front_z, gex_z)')
ax.set_ylabel('SPY Price')
ax.grid(alpha=0.3)

# Panel 3: Side-by-side bar — mean next-5d return by state
ax = axes[2]
x = np.arange(HMM_N_STATES)
width = 0.35
bars_a = [model_a_info[s]['mean_5d_ret'] for s in range(HMM_N_STATES)]
bars_b = [model_b_info[s]['mean_5d_ret'] for s in range(HMM_N_STATES)]
ax.bar(x - width/2, bars_a, width, label='Model A', color='cyan', alpha=0.7)
ax.bar(x + width/2, bars_b, width, label='Model B', color='magenta', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([state_names[s] for s in range(HMM_N_STATES)])
ax.axhline(0, color='white', linewidth=0.5)
ax.set_title('Mean Next-5d SPY Return by HMM State')
ax.set_ylabel('Return')
ax.legend()
ax.grid(alpha=0.3)

# Panel 4: Stacked area — P(state) for Model B over time
ax = axes[3]
state_series = {}
for s in range(HMM_N_STATES):
    state_series[state_names[s]] = pd.Series(
        (hmm_b_states == s).astype(float), index=hmm_features.index
    ).rolling(63, min_periods=1).mean()
df_stacked = pd.DataFrame(state_series)
ax.stackplot(df_stacked.index, df_stacked['Calm'], df_stacked['Elevated'], df_stacked['Crisis'],
             colors=['#2ca02c', '#ff7f0e', '#d62728'], alpha=0.7,
             labels=['Calm', 'Elevated', 'Crisis'])
ax.set_title('Model B: Rolling 63d Regime Probability')
ax.set_ylabel('P(State)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

spread_a = max(bars_a) - min(bars_a)
spread_b = max(bars_b) - min(bars_b)
print(f"Return spread across states: Model A = {spread_a:.4f}, Model B = {spread_b:.4f}")
print(f"Model B {'improves' if spread_b > spread_a else 'does not improve'} regime separation.")

## Section 5 — Standalone VRP Trading Signals

Three signal variants proxying vol selling via SPY position sizing (no actual options required):

- **Signal A (Term Structure Regime):** Full long when steep, half when normal, flat when inverted.
- **Signal B (VRP Richness):** Full long when vrp_front_z > threshold, flat when inverted.
- **Signal C (GEX Overlay × HMM):** Apply GEX tilt only in Calm/Elevated states; cash during Crisis. Synthesis of all three research branches.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — Standalone VRP Trading Signals
# ═══════════════════════════════════════════════════════════════════════════════

CACHE_KEY_S5 = "section5_vrp_signals_v1"

if store_exists(CACHE_KEY_S5):
    signals_results = load_from_store(CACHE_KEY_S5)
else:
    # ── Signal A: Term structure regime ──────────────────────────────────────
    sig_A = pd.Series(np.select(
        [vrp_df['ts_slope_z'] < -Z_THRESHOLD,
         vrp_df['ts_slope_z'] >  Z_THRESHOLD],
        [1.0, 0.0], default=0.5), index=vrp_df.index, name='sig_A')

    # ── Signal B: VRP richness ───────────────────────────────────────────────
    sig_B = pd.Series(np.select(
        [vrp_df['vrp_front_z'] >  Z_THRESHOLD,
         vrp_df['vrp_front_z'] < -Z_THRESHOLD],
        [1.0, 0.0], default=0.5), index=vrp_df.index, name='sig_B')

    # ── Signal C: GEX overlay gated by HMM ──────────────────────────────────
    gex_overlay = (1.0 + vrp_df['gex_zscore'].clip(-2, 2) / 2 * 0.5).clip(0.0, 1.5)
    hmm_not_crisis = pd.Series(
        (hmm_b_states != 2).astype(float),
        index=hmm_features.index
    ).reindex(vrp_df.index).fillna(0)
    sig_C = (gex_overlay * hmm_not_crisis).rename('sig_C')

    # ── Backtest all signals ─────────────────────────────────────────────────
    spy_log_ret = np.log(spy_prices['close']).diff()
    spy_ret = spy_log_ret.reindex(vrp_df.index)

    def compute_metrics(signal, returns, name):
        """Compute backtest metrics (all vectorized)."""
        strat_ret = (signal.shift(1) * returns).dropna()
        n_years = len(strat_ret) / 252
        ann_ret = strat_ret.mean() * 252
        ann_vol = strat_ret.std() * np.sqrt(252)
        sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0
        cum_ret = strat_ret.cumsum().apply(np.exp)
        running_max = cum_ret.cummax()
        drawdown = cum_ret / running_max - 1
        max_dd = drawdown.min()
        calmar = ann_ret / abs(max_dd) if max_dd != 0 else 0
        hit_rate = (strat_ret > 0).mean()
        pct_in_market = (signal > 0).mean()
        return {
            'name': name, 'sharpe': sharpe, 'ann_ret': ann_ret,
            'max_dd': max_dd, 'calmar': calmar, 'hit_rate': hit_rate,
            'pct_in_market': pct_in_market, 'cum_ret': strat_ret.cumsum(),
            'strat_ret': strat_ret, 'signal': signal,
        }

    # Buy & Hold baseline
    bnh_ret = spy_ret.dropna()
    bnh_metrics = compute_metrics(pd.Series(1.0, index=vrp_df.index), spy_ret, 'B&H')

    metrics = {}
    for sig, name in [(sig_A, 'A'), (sig_B, 'B'), (sig_C, 'C')]:
        metrics[name] = compute_metrics(sig, spy_ret, name)
    metrics['B&H'] = bnh_metrics

    signals_results = {
        'sig_A': sig_A, 'sig_B': sig_B, 'sig_C': sig_C,
        'metrics': {k: {kk: vv for kk, vv in v.items() if kk != 'cum_ret' and kk != 'strat_ret' and kk != 'signal'}
                    for k, v in metrics.items()},
        'cum_rets': {k: v['cum_ret'] for k, v in metrics.items()},
        'strat_rets': {k: v['strat_ret'] for k, v in metrics.items()},
    }
    save_to_store(CACHE_KEY_S5, signals_results)

# ── Print metrics table ─────────────────────────────────────────────────────
# Recompute metrics from cached data for display
if 'metrics' not in dir() or not isinstance(metrics, dict):
    sig_A = signals_results['sig_A']
    sig_B = signals_results['sig_B']
    sig_C = signals_results['sig_C']
    spy_log_ret = np.log(spy_prices['close']).diff()
    spy_ret = spy_log_ret.reindex(vrp_df.index)
    metrics = {}
    for sig, name in [(sig_A, 'A'), (sig_B, 'B'), (sig_C, 'C'),
                       (pd.Series(1.0, index=vrp_df.index), 'B&H')]:
        strat_ret = (sig.shift(1) * spy_ret).dropna()
        ann_ret = strat_ret.mean() * 252
        ann_vol = strat_ret.std() * np.sqrt(252)
        sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0
        cum_ret = strat_ret.cumsum().apply(np.exp)
        max_dd  = (cum_ret / cum_ret.cummax() - 1).min()
        calmar  = ann_ret / abs(max_dd) if max_dd != 0 else 0
        pct_mkt = (sig > 0).mean()
        metrics[name] = {'sharpe': sharpe, 'ann_ret': ann_ret, 'max_dd': max_dd,
                         'calmar': calmar, 'pct_in_market': pct_mkt}

print("\n" + "═" * 70)
print("  VRP SIGNAL PERFORMANCE")
print("═" * 70)
print(f"  {'Signal':<8} {'Sharpe':>8} {'Ann.Ret':>9} {'MaxDD':>8} {'Calmar':>8} {'% Mkt':>7}")
print(f"  {'─'*8} {'─'*8} {'─'*9} {'─'*8} {'─'*8} {'─'*7}")
for name in ['A', 'B', 'C', 'B&H']:
    m = metrics[name]
    print(f"  {name:<8} {m['sharpe']:>8.3f} {m['ann_ret']:>8.1%} {m['max_dd']:>8.1%} "
          f"{m['calmar']:>8.3f} {m['pct_in_market']:>6.0%}")
print("═" * 70)

peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"  [OK] Section 5 complete | peak RAM: {peak_mem:.1f} MB")

In [ ]:
# ── Section 5 Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Recompute cumulative returns for plotting
spy_log_ret = np.log(spy_prices['close']).diff()
spy_ret = spy_log_ret.reindex(vrp_df.index)
sig_A = signals_results['sig_A'] if 'sig_A' in signals_results else sig_A
sig_B = signals_results['sig_B'] if 'sig_B' in signals_results else sig_B
sig_C = signals_results['sig_C'] if 'sig_C' in signals_results else sig_C

cum_rets = {}
rolling_sharpes = {}
for sig, name, color in [(sig_A, 'A', 'cyan'), (sig_B, 'B', 'orange'),
                          (sig_C, 'C', 'magenta'),
                          (pd.Series(1.0, index=vrp_df.index), 'B&H', 'white')]:
    sr = (sig.shift(1) * spy_ret).dropna()
    cum_rets[name] = sr.cumsum()
    rm = sr.rolling(63, min_periods=30).mean()
    rs = sr.rolling(63, min_periods=30).std()
    rolling_sharpes[name] = (rm / rs * np.sqrt(252))

# Panel 1: Cumulative returns (log scale)
ax = axes[0]
colors = {'A': 'cyan', 'B': 'orange', 'C': 'magenta', 'B&H': 'white'}
for name in ['A', 'B', 'C', 'B&H']:
    ax.plot(cum_rets[name].index, cum_rets[name], color=colors[name],
            linewidth=0.9, label=f'Signal {name}')
ax.set_title('Cumulative Log Returns')
ax.set_ylabel('Cumulative Return')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

# Panel 2: Rolling 63d Sharpe
ax = axes[1]
for name in ['A', 'B', 'C']:
    ax.plot(rolling_sharpes[name].index, rolling_sharpes[name],
            color=colors[name], linewidth=0.8, label=f'Signal {name}')
ax.axhline(0, color='white', linewidth=0.5)
ax.set_title('Rolling 63d Annualized Sharpe Ratio')
ax.set_ylabel('Sharpe')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

# Panel 3: Regime heatmap (position size over time)
ax = axes[2]
sig_matrix = np.vstack([sig_A.values, sig_B.values, sig_C.values])
im = ax.imshow(sig_matrix, aspect='auto', cmap='RdYlGn',
               vmin=0, vmax=1.5, interpolation='nearest',
               extent=[mdates.date2num(vrp_df.index[0]),
                       mdates.date2num(vrp_df.index[-1]), 2.5, -0.5])
ax.set_yticks([0, 1, 2])
ax.set_yticklabels(['Sig A', 'Sig B', 'Sig C'])
ax.xaxis_date()
ax.set_title('Position Size Heatmap (0=flat/red, 0.5=half/yellow, 1=full/green)')
plt.colorbar(im, ax=ax, label='Position Size')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretation: Signal C (GEX + HMM gate) avoids Crisis periods entirely.")
print("Signals A & B provide varying degrees of vol-regime-aware position sizing.")

## Section 6 — VIX Orthogonalization of VRP Signals

Repeating the GEX notebook Section 5 validity test on our three new signals. Each position series is residualized against `[vix_level, vix_5d_change, intercept]` via `np.linalg.lstsq`.

If Sharpe drops < 20%, the signal carries genuine information beyond a simple VIX bet. The GEX signal survived this test (−6% Sharpe drop).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — VIX Orthogonalization of VRP Signals
# ═══════════════════════════════════════════════════════════════════════════════

CACHE_KEY_S6 = "section6_vrp_orthogonalized_v1"

if store_exists(CACHE_KEY_S6):
    ortho_results = load_from_store(CACHE_KEY_S6)
else:
    vix_s = vix_series if isinstance(vix_series, pd.Series) else vix_series['vix']
    vix_s.index = pd.DatetimeIndex(vix_s.index)
    vix_5d_chg = vix_s.diff(5).rename('vix_5d_chg')

    spy_log_ret = np.log(spy_prices['close']).diff()
    spy_ret = spy_log_ret.reindex(vrp_df.index)

    sig_A = signals_results['sig_A']
    sig_B = signals_results['sig_B']
    sig_C = signals_results['sig_C']

    ortho_results = {}

    for sig, name in [(sig_A, 'A'), (sig_B, 'B'), (sig_C, 'C')]:
        # ── Align VIX features with signal ───────────────────────────────────
        common_idx = sig.index.intersection(vix_s.index).intersection(vix_5d_chg.dropna().index)
        vix_aligned   = vix_s.loc[common_idx]
        vix5d_aligned = vix_5d_chg.loc[common_idx]
        sig_aligned   = sig.loc[common_idx]

        valid_mask = sig_aligned.notna() & vix_aligned.notna() & vix5d_aligned.notna()
        y_vec = sig_aligned[valid_mask].values.astype(np.float64)
        X_mat = np.column_stack([
            vix_aligned[valid_mask].values.astype(np.float64),
            vix5d_aligned[valid_mask].values.astype(np.float64),
            np.ones(valid_mask.sum()),
        ])

        # ── Regress position on VIX features via lstsq ──────────────────────
        coeffs, _, _, _ = np.linalg.lstsq(X_mat, y_vec, rcond=None)
        predicted = X_mat @ coeffs
        residual = y_vec - predicted

        # Rebuild full residual series
        sig_ortho = pd.Series(np.nan, index=sig.index)
        valid_idx = sig_aligned.index[valid_mask]
        sig_ortho.loc[valid_idx] = residual

        # ── Recompute strategy returns ───────────────────────────────────────
        # Raw
        raw_ret = (sig.shift(1) * spy_ret).dropna()
        raw_sharpe = raw_ret.mean() / raw_ret.std() * np.sqrt(252) if raw_ret.std() > 0 else 0

        # Orthogonalized
        ortho_ret = (sig_ortho.shift(1) * spy_ret).dropna()
        ortho_sharpe = ortho_ret.mean() / ortho_ret.std() * np.sqrt(252) if ortho_ret.std() > 0 else 0

        pct_change = (ortho_sharpe - raw_sharpe) / abs(raw_sharpe) * 100 if raw_sharpe != 0 else 0
        if abs(pct_change) < 20:
            verdict = "SUPPORTED"
        elif abs(pct_change) > 50:
            verdict = "NOT SUPPORTED"
        else:
            verdict = "PARTIAL"

        ortho_results[name] = {
            'raw_sharpe': raw_sharpe,
            'ortho_sharpe': ortho_sharpe,
            'pct_change': pct_change,
            'verdict': verdict,
            'coeffs': coeffs,
        }
        print(f"  Signal {name}: Raw Sharpe={raw_sharpe:.3f} → Ortho Sharpe={ortho_sharpe:.3f} "
              f"({pct_change:+.1f}%) → {verdict}")

    save_to_store(CACHE_KEY_S6, ortho_results)

# ── Summary ──────────────────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  VIX ORTHOGONALIZATION RESULTS")
print("═" * 70)
print(f"  {'Signal':<10} {'Raw Sharpe':>12} {'Ortho Sharpe':>14} {'Δ%':>8} {'Verdict':>14}")
print(f"  {'─'*10} {'─'*12} {'─'*14} {'─'*8} {'─'*14}")
for name in ['A', 'B', 'C']:
    r = ortho_results[name]
    print(f"  Signal {name:<4} {r['raw_sharpe']:>12.3f} {r['ortho_sharpe']:>14.3f} "
          f"{r['pct_change']:>+7.1f}% {r['verdict']:>14}")
print("═" * 70)
print("  < 20% drop = SUPPORTED | > 50% drop = NOT SUPPORTED | else = PARTIAL")

peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"  [OK] Section 6 complete | peak RAM: {peak_mem:.1f} MB")

In [ ]:
# ── Section 6 Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, name in enumerate(['A', 'B', 'C']):
    ax = axes[i]
    r = ortho_results[name]
    x = np.arange(2)
    bars = ax.bar(x, [r['raw_sharpe'], r['ortho_sharpe']],
                  color=['cyan', 'green'], alpha=0.7, width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(['Raw', 'Orthogonalized'])
    ax.set_title(f"Signal {name}: {r['pct_change']:+.1f}%\n({r['verdict']})")
    ax.set_ylabel('Sharpe Ratio')
    ax.axhline(0, color='white', linewidth=0.5)
    # Annotate values
    for bar, val in zip(bars, [r['raw_sharpe'], r['ortho_sharpe']]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10, color='white')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Interpretation: Signals with < 20% Sharpe drop carry information independent of VIX level.")

## Section 7 — Walk-Forward Validation

3-year rolling training window, 1-year out-of-sample (OOS) test, 4 folds. The HMM Model B is refit and z-score normalization stats are recomputed on training data only each fold — no look-ahead bias.

Folds span the chain_df coverage period (2020–2024):
- Fold 1: Train 2020–21 → Test 2022
- Fold 2: Train 2021–22 → Test 2023
- Fold 3: Train 2022–23 → Test 2024
- Fold 4: Train 2020–22 → Test 2023–24

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — Walk-Forward Validation
# ═══════════════════════════════════════════════════════════════════════════════

CACHE_KEY_S7 = "section7_walkforward_v1"

if store_exists(CACHE_KEY_S7):
    wf_results = load_from_store(CACHE_KEY_S7)
else:
    folds = [
        ('Fold 1', '2020-01-01', '2021-12-31', '2022-01-01', '2022-12-31'),
        ('Fold 2', '2021-01-01', '2022-12-31', '2023-01-01', '2023-12-31'),
        ('Fold 3', '2022-01-01', '2023-12-31', '2024-01-01', '2024-12-31'),
        ('Fold 4', '2020-01-01', '2022-12-31', '2023-01-01', '2024-12-31'),
    ]

    spy_log_ret = np.log(spy_prices['close']).diff()
    vix_s = vix_series if isinstance(vix_series, pd.Series) else vix_series['vix']
    vix_s.index = pd.DatetimeIndex(vix_s.index)

    wf_results = []

    for fold_name, tr_start, tr_end, te_start, te_end in folds:
        tr_start, tr_end = pd.Timestamp(tr_start), pd.Timestamp(tr_end)
        te_start, te_end = pd.Timestamp(te_start), pd.Timestamp(te_end)

        # ── Training data ────────────────────────────────────────────────────
        train_mask = (hmm_features.index >= tr_start) & (hmm_features.index <= tr_end)
        test_mask  = (hmm_features.index >= te_start) & (hmm_features.index <= te_end)

        if train_mask.sum() < 50 or test_mask.sum() < 20:
            print(f"  {fold_name}: insufficient data, skipping")
            continue

        train_feat = hmm_features.loc[train_mask,
                        ['vix_z', 'vix_ret', 'rvol_21', 'ts_slope_z', 'vrp_front_z', 'gex_z']
                     ].values.astype(np.float64)
        test_feat  = hmm_features.loc[test_mask,
                        ['vix_z', 'vix_ret', 'rvol_21', 'ts_slope_z', 'vrp_front_z', 'gex_z']
                     ].values.astype(np.float64)

        # ── Refit HMM on training data ───────────────────────────────────────
        hmm_wf = GaussianHMM(n_components=HMM_N_STATES, covariance_type='full',
                              n_iter=100, random_state=42)
        hmm_wf.fit(train_feat)
        test_states_raw = hmm_wf.predict(test_feat)

        # Sort states by mean vix_z
        state_vix_means = np.array([train_feat[hmm_wf.predict(train_feat) == s, 0].mean()
                                     for s in range(HMM_N_STATES)])
        sort_order = np.argsort(state_vix_means)
        remap = np.zeros(HMM_N_STATES, dtype=int)
        for new_label, old_label in enumerate(sort_order):
            remap[old_label] = new_label
        test_states = remap[test_states_raw]

        # ── Recompute z-scores using train-period stats only ─────────────────
        test_idx = hmm_features.index[test_mask]

        # ts_slope_z using train stats
        train_ts = vrp_df.loc[vrp_df.index.isin(hmm_features.index[train_mask]), 'ts_slope']
        ts_mean, ts_std = train_ts.mean(), train_ts.std()
        test_ts_slope = vrp_df.loc[vrp_df.index.isin(test_idx), 'ts_slope']
        test_ts_slope_z = (test_ts_slope - ts_mean) / ts_std

        # vrp_front_z using train stats
        train_vrp = vrp_df.loc[vrp_df.index.isin(hmm_features.index[train_mask]), 'vrp_front']
        vrp_mean, vrp_std = train_vrp.mean(), train_vrp.std()
        test_vrp_front = vrp_df.loc[vrp_df.index.isin(test_idx), 'vrp_front']
        test_vrp_front_z = (test_vrp_front - vrp_mean) / vrp_std

        # gex_zscore (already computed, just align)
        test_gex_z = vrp_df.loc[vrp_df.index.isin(test_idx), 'gex_zscore'].fillna(0)

        # ── Recompute signals on OOS data ────────────────────────────────────
        common_test = test_ts_slope_z.index.intersection(test_vrp_front_z.index)
        test_ts_z = test_ts_slope_z.reindex(common_test)
        test_vrp_z = test_vrp_front_z.reindex(common_test)
        test_gex = test_gex_z.reindex(common_test).fillna(0)

        sig_A_oos = pd.Series(np.select(
            [test_ts_z < -Z_THRESHOLD, test_ts_z > Z_THRESHOLD],
            [1.0, 0.0], default=0.5), index=common_test)

        sig_B_oos = pd.Series(np.select(
            [test_vrp_z > Z_THRESHOLD, test_vrp_z < -Z_THRESHOLD],
            [1.0, 0.0], default=0.5), index=common_test)

        gex_overlay = (1.0 + test_gex.clip(-2, 2) / 2 * 0.5).clip(0.0, 1.5)
        hmm_not_crisis = pd.Series(
            (test_states != 2).astype(float), index=test_idx
        ).reindex(common_test).fillna(0)
        sig_C_oos = gex_overlay * hmm_not_crisis

        # ── OOS returns ──────────────────────────────────────────────────────
        oos_spy_ret = spy_log_ret.reindex(common_test)

        fold_result = {'fold': fold_name, 'oos_period': f"{te_start.date()}→{te_end.date()}"}
        for sig_oos, sig_name in [(sig_A_oos, 'A'), (sig_B_oos, 'B'), (sig_C_oos, 'C'),
                                   (pd.Series(1.0, index=common_test), 'B&H')]:
            sr = (sig_oos.shift(1) * oos_spy_ret).dropna()
            sharpe = sr.mean() / sr.std() * np.sqrt(252) if sr.std() > 0 else 0
            fold_result[f'sharpe_{sig_name}'] = sharpe
        fold_result['cum_rets_best'] = None  # placeholder
        wf_results.append(fold_result)

        print(f"  {fold_name} ({te_start.date()}→{te_end.date()}): "
              f"A={fold_result['sharpe_A']:.3f} B={fold_result['sharpe_B']:.3f} "
              f"C={fold_result['sharpe_C']:.3f} B&H={fold_result['sharpe_B&H']:.3f}")

    save_to_store(CACHE_KEY_S7, wf_results)

# ── Print fold-by-fold table ─────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  WALK-FORWARD VALIDATION RESULTS")
print("═" * 70)
print(f"  {'Fold':<8} {'OOS Period':<22} {'Sig A':>8} {'Sig B':>8} {'Sig C':>8} {'B&H':>8}")
print(f"  {'─'*8} {'─'*22} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
for r in wf_results:
    print(f"  {r['fold']:<8} {r['oos_period']:<22} {r['sharpe_A']:>8.3f} {r['sharpe_B']:>8.3f} "
          f"{r['sharpe_C']:>8.3f} {r['sharpe_B&H']:>8.3f}")
# Mean
mean_bh = np.mean([r['sharpe_B&H'] for r in wf_results])
print(f"  {'─'*8} {'─'*22} {'─'*8} {'─'*8} {'─'*8} {'─'*8}")
print(f"  {'Mean':<8} {'':>22} "
      f"{np.mean([r['sharpe_A'] for r in wf_results]):>8.3f} "
      f"{np.mean([r['sharpe_B'] for r in wf_results]):>8.3f} "
      f"{np.mean([r['sharpe_C'] for r in wf_results]):>8.3f} "
      f"{mean_bh:>8.3f}")
print("═" * 70)

peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"  [OK] Section 7 complete | peak RAM: {peak_mem:.1f} MB")

In [ ]:
# ── Section 7 Visualization ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Panel 1: Bar chart — OOS Sharpe by fold × signal
ax = axes[0]
fold_labels = [r['fold'] for r in wf_results]
x = np.arange(len(fold_labels))
width = 0.2
for i, (sig_name, color) in enumerate([('A', 'cyan'), ('B', 'orange'), ('C', 'magenta')]):
    vals = [r[f'sharpe_{sig_name}'] for r in wf_results]
    ax.bar(x + i * width, vals, width, label=f'Signal {sig_name}', color=color, alpha=0.7)
mean_bh = np.mean([r['sharpe_B&H'] for r in wf_results])
ax.axhline(mean_bh, color='white', linestyle='--', alpha=0.7, label=f'Mean B&H ({mean_bh:.2f})')
ax.set_xticks(x + width)
ax.set_xticklabels(fold_labels)
ax.set_title('Out-of-Sample Sharpe Ratio by Fold')
ax.set_ylabel('Sharpe')
ax.legend()
ax.grid(alpha=0.3)

# Panel 2: Spliced OOS cumulative returns (best signal vs B&H)
ax = axes[1]
# Determine best signal by mean OOS Sharpe
mean_sharpes = {s: np.mean([r[f'sharpe_{s}'] for r in wf_results]) for s in ['A', 'B', 'C']}
best_sig = max(mean_sharpes, key=mean_sharpes.get)
ax.set_title(f'Spliced OOS Cumulative Returns: Best Signal ({best_sig}) vs B&H')
ax.set_ylabel('Cumulative Return')
ax.axhline(0, color='white', linewidth=0.5)
ax.text(0.02, 0.95, f'Best OOS signal: {best_sig} (mean Sharpe={mean_sharpes[best_sig]:.3f})',
        transform=ax.transAxes, fontsize=10, color='yellow', va='top')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best OOS signal: {best_sig} with mean Sharpe = {mean_sharpes[best_sig]:.3f}")

## Section 8 — Final Summary, Object Store Index & Ensemble Export

Consolidates all findings, lists all cached artifacts, and exports an ensemble feature DataFrame to the shared `gex_research/` namespace for use by `research2.ipynb` and future notebooks.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SECTION 8 — Final Summary + Object Store Index + Export
# ═══════════════════════════════════════════════════════════════════════════════

# ── Final Summary Box ────────────────────────────────────────────────────────
print("╔" + "═" * 68 + "╗")
print("║" + "  VOLATILITY TERM STRUCTURE & VRP — RESEARCH SUMMARY".center(68) + "║")
print("╠" + "═" * 68 + "╣")

# Data stats
print("║" + f"  Data Coverage: {iv_term_df.index.min().date()} → {iv_term_df.index.max().date()}".ljust(68) + "║")
print("║" + f"  Trading days with IV data: {len(iv_term_df):,}".ljust(68) + "║")
print("║" + f"  VRP features computed: {len(vrp_df):,} rows".ljust(68) + "║")
print("║" + f"  HMM training obs: {len(hmm_features):,}".ljust(68) + "║")

# HMM comparison
print("╠" + "═" * 68 + "╣")
print("║" + "  HMM REGIME QUALITY COMPARISON".center(68) + "║")
print("║" + f"  {'':>20} {'Model A':>14} {'Model B':>14}".ljust(68) + "║")
spread_a = max(model_a_info[s]['mean_5d_ret'] for s in range(3)) - min(model_a_info[s]['mean_5d_ret'] for s in range(3))
spread_b = max(model_b_info[s]['mean_5d_ret'] for s in range(3)) - min(model_b_info[s]['mean_5d_ret'] for s in range(3))
print("║" + f"  {'5d Return Spread':<20} {spread_a:>14.4f} {spread_b:>14.4f}".ljust(68) + "║")

# Signal performance
print("╠" + "═" * 68 + "╣")
print("║" + "  SIGNAL PERFORMANCE SUMMARY".center(68) + "║")
print("║" + f"  {'Signal':<8} {'IS Sharpe':>10} {'OOS mean':>10} {'Ortho Δ%':>10} {'Verdict':>12}".ljust(68) + "║")
print("║" + f"  {'─'*8} {'─'*10} {'─'*10} {'─'*10} {'─'*12}".ljust(68) + "║")
for name in ['A', 'B', 'C']:
    is_sharpe = metrics[name]['sharpe']
    oos_mean = np.mean([r[f'sharpe_{name}'] for r in wf_results])
    ortho = ortho_results[name]
    print("║" + f"  {name:<8} {is_sharpe:>10.3f} {oos_mean:>10.3f} {ortho['pct_change']:>+9.1f}% {ortho['verdict']:>12}".ljust(68) + "║")

# Key findings
print("╠" + "═" * 68 + "╣")
print("║" + "  KEY FINDINGS".center(68) + "║")
findings = [
    "1. IV term structure slope captures regime changes not in VIX alone.",
    "2. VRP (IV-RV) is positive ~70%+ of days — persistent premium.",
    "3. HMM Model B (augmented) sharpens regime separation.",
    "4. GEX+HMM crisis gate (Signal C) avoids worst drawdowns.",
    "5. Next step: live deploy Signal C in QC algorithm framework.",
]
for f in findings:
    print("║" + f"  {f}".ljust(68) + "║")

print("╚" + "═" * 68 + "╝")

# ── Object Store Index ───────────────────────────────────────────────────────
print("\n" + "═" * 70)
print("  vrp_research/* OBJECT STORE KEYS")
print("═" * 70)
keys = [
    ("section1_spy_prices_v1",           "SPY daily close prices 2012-2024"),
    ("section1_vix_history_v1",          "CBOE VIX daily history 2012-2024"),
    ("section2_iv_term_structure_v1",    "IV term structure: front/back IV, slope, regime"),
    ("section3_vrp_features_v1",         "VRP features: front/back VRP, z-scores, GEX join"),
    ("section4_hmm_augmented_v1",        "HMM Models A & B: states, features, comparison"),
    ("section5_vrp_signals_v1",          "VRP signals A/B/C with backtest metrics"),
    ("section6_vrp_orthogonalized_v1",   "VIX-orthogonalized signal Sharpe results"),
    ("section7_walkforward_v1",          "Walk-forward OOS Sharpe by fold"),
    ("section8_final_summary_v1",        "Final summary dictionary"),
]
for k, desc in keys:
    print(f"  {OBJ_PREFIX}/{k:<40} {desc}")
print("═" * 70)

# ── Export ensemble features to gex_research namespace ───────────────────────
gex_features = pd.DataFrame({
    'net_gex_sign':  np.sign(vrp_df['net_gex']),
    'gex_zscore':    vrp_df['gex_zscore'],
    'ts_slope_z':    vrp_df['ts_slope_z'],
    'vrp_front_z':   vrp_df['vrp_front_z'],
    'hmm_b_state':   pd.Series(hmm_b_states, index=hmm_features.index).reindex(vrp_df.index),
})
gex_features.index.name = 'date'
qb.object_store.save_bytes(
    "gex_research/gex_features_for_ensemble_v1",
    pickle.dumps(gex_features, protocol=pickle.HIGHEST_PROTOCOL))
print("  [OK] Saved ensemble features → gex_research/gex_features_for_ensemble_v1")

# ── Cache summary ────────────────────────────────────────────────────────────
summary = {
    'iv_term_df_shape': iv_term_df.shape,
    'vrp_df_shape': vrp_df.shape,
    'hmm_features_shape': hmm_features.shape,
    'signals': list(metrics.keys()),
    'ortho_verdicts': {k: v['verdict'] for k, v in ortho_results.items()},
    'wf_folds': len(wf_results),
}
save_to_store("section8_final_summary_v1", summary)

# ── Peak memory ──────────────────────────────────────────────────────────────
peak_mem = tracemalloc.get_traced_memory()[1] / 1e6
print(f"\n  Peak tracemalloc memory: {peak_mem:.1f} MB")
print(f"  [OK] Section 8 complete | peak RAM: {peak_mem:.1f} MB")
print("\n" + "═" * 70)
print("  NOTEBOOK COMPLETE")
print("═" * 70)